# Module 2: Vector Search (Lessons 1-7)

In Module 1, search matched **exact words**. Here we search by **meaning**.

| Lesson | Topic |
| --- | --- |
| 1 | What is vector search |
| 2 | Embeddings |
| 3 | Embedding our dataset |
| 4 | Vector search with numpy |
| 5 | Vector search with minsearch |
| 6 | RAG with vector search |
| 7 | Vector search with sqlitesearch (part A here, part B in `07b_sqlitesearch_persistent.ipynb`) |

Next: `07b_sqlitesearch_persistent.ipynb`, then `08_pgvector.ipynb`.

---
# Lesson 1: What is vector search

**Simple version:** Keyword search is like finding a book only by the exact words on its cover. If you ask "can I still **enroll**?" but the FAQ says "can I still **join**?", keyword search misses it. **Vector search** turns every text into a list of numbers (a **vector**) that describes its *meaning*. Texts that mean similar things get similar numbers, even with different words. First we turn all FAQ cards into vectors (once, offline). Then, for each question, we find the closest cards.

**End of Lesson 1.** No notebook code here (only project setup). The idea: **match meaning, not exact words.**

---

---
# Lesson 2: Embeddings

**Simple version:** An **embedding model** is a translator from words to numbers. We use `all-MiniLM-L6-v2`, which turns any text into 384 numbers. Think of it as a spot on a giant map: texts with similar meanings land close together.

Two questions that mean the same thing but share almost no words. Keyword search would struggle here.

In [44]:
q1 = "I just discovered the course, can I still join?"

q2 = "I just found out about the program, can I still enroll?"

Load the embedding model (the words-to-numbers translator).

In [45]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Turn each question into a vector. `shape` `(384,)` means 384 numbers.

In [46]:
v1 = model.encode(q1)
v1.shape

(384,)

In [47]:
v2 = model.encode(q2)
v2.shape

(384,)

A new question, and an FAQ answer about registering.

In [48]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [49]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

**Dot product** = how alike two vectors are. Bigger number = closer meaning.

In [50]:
v1.dot(dv)

np.float32(0.32332402)

A question about something else (Docker). Its score with the same answer should be much lower.

In [51]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [52]:
v2.dot(dv)

np.float32(0.019730477)

**End of Lesson 2.** Similar meaning gives a high score, a different topic gives a low score.

---

---
# Lesson 3: Embedding our dataset

**Simple version:** Now we translate **every** FAQ card into a vector, so later we can compare a question against all of them.

Download all the FAQ cards (same helper as Module 1).

In [53]:
from ingest import load_faq_data

documents = load_faq_data()

For each card, glue the question and the answer into one text.

In [54]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

Turn the texts into vectors, 50 at a time (a batch). `tqdm` shows a progress bar.

In [55]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/30 [00:00<?, ?it/s]

1470

Stack all the vectors into one big table `X`: one row per card.

In [56]:
import numpy as np
X = np.array(vectors)

**End of Lesson 3.** Every FAQ card now has a vector. `X` is our "meaning map" of the whole FAQ.

---

---
# Lesson 4: Vector search with numpy

**Simple version:** Searching is just math now: compare the question's vector with every card's vector and pick the biggest scores. With numpy we score all cards in one line.

Score every card at once (table `X` times the question vector `v1`).

In [57]:
scores = X.dot(v1)

Where is the best score, and how big is it?

In [58]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(1074), np.float32(0.762941))

Look at that winning card.

In [59]:
documents[1074]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

The top 5: `argsort` sorts scores from small to big, so we take the last 5.

In [60]:
top5 = np.argsort(scores)[-5:]

In [61]:
top5

array([1079,    0,  190,  732, 1074])

Print the top 5 cards with their scores.

In [62]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.5600999
{'id': '068529125b', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}

0.6536312
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date,

**End of Lesson 4.** We built a working search engine with just numpy: score, sort, pick the top.

---

---
# Lesson 5: Vector search with minsearch

**Simple version:** Same search as Lesson 4, but packaged in a ready-made tool (`minsearch.VectorSearch`). It also lets us **filter**, for example "only llm-zoomcamp cards".

Build the vector index from our vectors `X` and the cards.

In [63]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

Search with the question vector, only on the llm-zoomcamp shelf.

In [64]:
results = vindex.search(
    v1,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

**End of Lesson 5.** Vector search in two short calls: `fit()` and `search()`.

---

---
# Lesson 6: RAG with vector search

**Simple version:** Take the Module 1 RAG robot and swap its "eyes": instead of matching words, it now matches meaning. Everything else (prompt, LLM) stays the same.

Phone line to the AI (Azure, gpt-5-nano).

In [65]:
from dotenv import load_dotenv
from openai import OpenAI
from os import getenv

load_dotenv()
openai_client = OpenAI(
    base_url="https://26120057-2478-resource.openai.azure.com/openai/v1",
    api_key=getenv("AZURE_API_KEY"),
)

The old keyword index from Module 1, so we can compare.

In [66]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

The old keyword robot.

In [67]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

Ask with different words ("found out", "sign up"). Keyword search may not find the right card.

In [68]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can sign up. If you want to receive a certificate, you’ll need to submit your project while submissions are still open.'

`RAGVector`: the same robot blueprint as Module 1's `RAGBase`, but with a new `search()`. This is **inheritance**: copy a blueprint and change just one part. Now search turns the question into a vector and looks for the closest card vectors.

In [69]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

Build the vector robot: the embedding model + the vector index + the AI client.

In [70]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

Ask with different words again. Vector search finds the right card by meaning.

In [71]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes. You can still join—the form is mainly to gauge interest. You don’t strictly need to register to participate; you can start learning and submit homework while the form is open.\n\nIf you want a certificate, you’ll need to submit your capstone project while submissions are still being accepted. Certificates are issued after all submissions and peer reviews are complete.'

**End of Lesson 6.** Our RAG now understands questions asked in different words.

---

---
# Lesson 7 (part A): Vector search with sqlitesearch: save the vectors

**Simple version:** Embedding thousands of cards takes time. Like M1's `faq.db`, we save the vectors once in a file (`faq_vectors2.db`) so we never redo the work. `mode="ivf"` sorts the vectors into groups of similar ones, so a search only checks the nearby groups instead of every card (faster).

Create (or open) the vector database file.

In [72]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

Save all the vectors and their cards into the file.

In [73]:
vs_index.fit(vectors, documents)

Try a search on the saved index.

In [74]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

Close the database so everything is written to disk.

In [76]:
vs_index.close()

**End of Lesson 7 (part A).** `faq_vectors2.db` holds all the vectors.

Next: `07b_sqlitesearch_persistent.ipynb` uses it.

---